In [ ]:
logger.info("\\n" + "="*60)
logger.info("PIPELINE 01 SUMMARY")
logger.info("="*60)
logger.info(f"Preprocessed training images: {X_train.shape}")
logger.info(f"Preprocessed test images: {X_test.shape}")
logger.info(f"Labels: {y_train.shape}")
logger.info(f"CV Folds: {config.train_val_split.num_splits}")
logger.info(f"Augmentation: {config.augmentation.enabled}")
logger.info("\\nNext step: Run Notebook 02 - Model Training")
logger.info("="*60 + "\\n")

print("✓ Pipeline 01 completed successfully!")

## 7. Summary

In [ ]:
logger.info("Saving preprocessed data...")

# Save numpy arrays
np.save('processed/X_train.npy', X_train)
np.save('processed/X_test.npy', X_test)
np.save('processed/y_train.npy', y_train)

# Save fold metadata
splitter.save_fold_metadata(fold_metadata, 'processed/fold_metadata.json')

logger.info("✓ Saved X_train.npy")
logger.info("✓ Saved X_test.npy")
logger.info("✓ Saved y_train.npy")
logger.info("✓ Saved fold_metadata.json")

print("\\n--- Files Saved ---")
print(f"  processed/X_train.npy ({X_train.shape})")
print(f"  processed/X_test.npy ({X_test.shape})")
print(f"  processed/y_train.npy ({y_train.shape})")
print(f"  processed/fold_metadata.json ({len(fold_metadata)} folds)")

## 6. Save Preprocessed Data and Fold Metadata

In [ ]:
# Verify stratification for first fold
fold_0_info = fold_metadata[0]
train_idx = np.array(fold_0_info['train_indices'])
val_idx = np.array(fold_0_info['val_indices'])

print("\\nStratification Check (Fold 0):")
print(f"  Train class counts: {fold_0_info['train_class_counts']}")
print(f"  Val class counts: {fold_0_info['val_class_counts']}")

# Verify no overlap between train and val
assert len(set(train_idx) & set(val_idx)) == 0, "Train/Val indices overlap detected!"
assert len(train_idx) + len(val_idx) == len(y_train), "Indices count mismatch!"

print("  ✓ Stratification verified - no overlap, correct counts")

## 5. Verify Stratification

## 4. Extract Labels and Create CV Splits

In [ ]:
logger.info("Preprocessing test images...")
X_test = preprocessor.preprocess_batch(test_df['full_path'].tolist())

print(f"Preprocessed test data shape: {X_test.shape}")
print(f"Memory usage: {X_test.nbytes / 1e6:.1f} MB")

## 3. Preprocess Test Images

In [ ]:
logger.info("Preprocessing training images...")
X_train = preprocessor.preprocess_batch(train_df['full_path'].tolist())

print(f"Preprocessed training data shape: {X_train.shape}")
print(f"Data type: {X_train.dtype}")
print(f"Value range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Memory usage: {X_train.nbytes / 1e6:.1f} MB")

## 2. Preprocess Training Images

In [ ]:
# Initialize preprocessor
preprocessor = ImagePreprocessor(
    target_height=config.data.target_height,
    target_width=config.data.target_width,
    normalization=config.preprocessing.normalization,
    color_mode=config.data.color_mode
)

# Initialize augmenter
augmenter = ImageAugmenter(
    enabled=config.augmentation.enabled,
    horizontal_flip_p=config.augmentation.horizontal_flip_p,
    rotation_degrees=config.augmentation.rotation_degrees,
    brightness_factor=config.augmentation.brightness_factor,
    contrast_factor=config.augmentation.contrast_factor,
)

logger.info(f"Preprocessor: {config.data.target_height}x{config.data.target_width}, {config.preprocessing.normalization} normalization")
logger.info(f"Augmenter: enabled={config.augmentation.enabled}")

## 1. Initialize Preprocessing & Augmentation Pipelines

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from config_loader import load_config
from logger_setup import setup_logger
from preprocessor import ImagePreprocessor, ImageAugmenter
from train_val_split import TrainValSplitter
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

logger = setup_logger('pipeline_01', level='INFO')
logger.info("Pipeline 01: Preprocessing, Augmentation & Splits - Started")

# Load config and data
config = load_config()
train_df = pd.read_parquet('processed/train_metadata.parquet')
test_df = pd.read_parquet('processed/test_metadata.parquet')

logger.info(f"Loaded {len(train_df)} training samples and {len(test_df)} test samples")

# Synthetic Image Attribution Challenge - Pipeline 1
## Stage 01: Preprocessing, Augmentation & Train/Validation Splits

This notebook preprocesses images, creates augmentation pipelines, and generates stratified K-fold cross-validation splits.

**Expected Duration:** ~20-30 minutes (plus preprocessing time)  
**Inputs:** `processed/train_metadata.parquet`, `processed/test_metadata.parquet`  
**Outputs:**
- `processed/X_train.npy` - Preprocessed training images
- `processed/X_test.npy` - Preprocessed test images
- `processed/y_train.npy` - Training labels
- `processed/fold_metadata.json` - CV fold assignments
- `logs/pipeline_01.log` - Detailed logs